In [1]:
# Cell 1: Mount Drive, locate baseline, set CWD, check GPU
import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

def find_baseline_dir():
    candidates = [
        "/content/drive/MyDrive/final_project/baseline",
        "/content/drive/MyDrive/final_project/baseline/",
    ]
    for p in candidates:
        if os.path.isdir(p):
            return os.path.abspath(p)

    shared_root = "/content/drive/Shareddrives"
    if os.path.isdir(shared_root):
        for root, dirs, _ in os.walk(shared_root):
            if root.endswith("/final_project") and "baseline" in dirs:
                return os.path.abspath(os.path.join(root, "baseline"))

    raise FileNotFoundError("Could not find final_project/baseline in Drive.")

BASE_DIR = find_baseline_dir()
os.chdir(BASE_DIR)

print("BASE_DIR =", BASE_DIR)
print("CWD =", os.getcwd())

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
BASE_DIR = /content/drive/MyDrive/final_project/baseline
CWD = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline
CUDA available: True
GPU: NVIDIA L4


In [2]:
# Cell 2: Install pinned deps
import sys, subprocess

# Upgrade tooling
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "-q"])

# Remove conflicts (ignore if missing)
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y",
                 "transformers", "tokenizers", "huggingface-hub",
                 "pandas", "numpy", "tqdm", "pyyaml"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

pkgs = [
    "pyyaml==6.0.1",
    "tqdm==4.66.2",
    "numpy==1.26.4",
    "pandas==2.2.2",
    "huggingface-hub==0.21.4",
    "tokenizers==0.15.2",
    "transformers==4.38.1",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

import yaml, tqdm, numpy, pandas, transformers, huggingface_hub, tokenizers
print("huggingface_hub:", huggingface_hub.__version__)
print("tokenizers:", tokenizers.__version__)
print("transformers:", transformers.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("Installed OK")

huggingface_hub: 0.21.4
tokenizers: 0.15.2
transformers: 4.38.1
numpy: 1.26.4
pandas: 2.2.2
Installed OK


In [3]:
# Cell 3: Create a tiny dummy mlx_lm package
import os

# quantize_mistral_mlx.py does: from mlx_lm.utils import convert
# On Colab, mlx_lm is not available (it's mac-focused). We stub it.
mlx_root = os.path.join(BASE_DIR, "mlx_lm")
os.makedirs(mlx_root, exist_ok=True)

with open(os.path.join(mlx_root, "__init__.py"), "w") as f:
    f.write("# dummy mlx_lm package for Colab\n")

with open(os.path.join(mlx_root, "utils.py"), "w") as f:
    f.write(
        "def convert(*args, **kwargs):\n"
        "    raise RuntimeError('mlx_lm.convert is not supported on this environment (dummy stub).')\n"
    )

print("Created dummy mlx_lm at:", mlx_root)

Created dummy mlx_lm at: /content/drive/MyDrive/final_project/baseline/mlx_lm


In [4]:
# Cell 4 (NEW): Clone the repo (fast local clone), keep outputs on Drive
import os, subprocess

REPO_URL = "https://github.com/ali-mohmmadi/KGP-CuriousLLM.git"
REPO_DIR = "/content/KGP-CuriousLLM"

if not os.path.isdir(REPO_DIR):
    print("Cloning repository into:", REPO_DIR)
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print("Repo already exists. Pulling latest changes...")
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])

print("Repo ready at:", REPO_DIR)
print("Repo root files:", os.listdir(REPO_DIR)[:10])

Cloning repository into: /content/KGP-CuriousLLM
Repo ready at: /content/KGP-CuriousLLM
Repo root files: ['kgp_main.py', 'T5_main.py', 'configs', '.gitignore', 'README.md', 'quantize_mistral_main.py', 'requirements.txt', 'MDR_main.py', 'KGP', 'create_dirs.py']


In [5]:
# Cell 5 (REPLACE): Add repo + baseline to PYTHONPATH, then import
import sys, os

REPO_DIR = "/content/KGP-CuriousLLM"

# 1) baseline (Drive) first so dummy mlx_lm can be found if needed
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

# 2) repo root so `import KGP...` works
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Smoke test imports (repo-faithful)
from KGP.MDR.tokenizer import load_tokenizer
from KGP.KG.mdr_encoder import Retriever_inf
from KGP.KG.train import run
from KGP.LLMs.Mistral.quantize_mistral_mlx import load_config

print("Imports OK")
print("KGP package loaded from:", REPO_DIR)

Imports OK
KGP package loaded from: /content/KGP-CuriousLLM


In [6]:
# Cell 6: Copy your new dataset into baseline (repo-style path)
import os, shutil, json

src = "/content/drive/MyDrive/final_project/hotpotqa_dev_2017wiki_1000_converted.json"
dst_dir = os.path.join(BASE_DIR, "DATA", "HotpotQA")
os.makedirs(dst_dir, exist_ok=True)

dst = os.path.join(dst_dir, "hotpotqa_dev_2017wiki_1000_converted.json")

assert os.path.isfile(src), f"Missing source dataset: {src}"

if not os.path.isfile(dst):
    shutil.copyfile(src, dst)

print("Dataset in baseline:", dst)

# Small sanity check
data = json.load(open(dst, "r"))
print("num_records =", len(data))
print("keys(example[0]) =", list(data[0].keys()))
print("title_chunks_len(example[0]) =", len(data[0].get("title_chunks", [])))

Dataset in baseline: /content/drive/MyDrive/final_project/baseline/DATA/HotpotQA/hotpotqa_dev_2017wiki_1000_converted.json
num_records = 1000
keys(example[0]) = ['question', 'answer', 'type', 'titles', 'docs_chunks', 'docs', 'title_chunks', 'supports']
title_chunks_len(example[0]) = 467


In [7]:
# Cell 7 : Write Hotpot config + also mirror it into mdr_2wiki (repo script compatibility)
import os, yaml, shutil
import torch

cfg_dir = os.path.join(BASE_DIR, "configs", "mdr_embedding")
os.makedirs(cfg_dir, exist_ok=True)

cfg_hotpot = os.path.join(cfg_dir, "mdr_hotpot_embedding.yml")
cfg_2wiki  = os.path.join(cfg_dir, "mdr_2wiki_embedding.yml")  # keep for MDR_embedding_main.py compatibility

# Backup (optional)
for p in [cfg_hotpot, cfg_2wiki]:
    bak = p + ".bak"
    if os.path.isfile(p) and not os.path.isfile(bak):
        shutil.copyfile(p, bak)
        print("Backed up:", bak)

device = "cuda" if torch.cuda.is_available() else "cpu"

args_dict = {
    "root_dir": ".",
    "dataset": "DATA/HotpotQA/hotpotqa_dev_2017wiki_1000_converted.json",
    "model": {
        "run_id": "hotpotqa_dev2017wiki_1000_old",
        "base_model": "bert-base-uncased",
        "from_checkpoint": "mdr_best_model_old.pt",
        "batch_size": 1,
        "max_token_len": 200,
        "device": device,
        "save_every": 250000,
    },
    "emb_checkpoint": "",
}

# Write Hotpot config
with open(cfg_hotpot, "w") as f:
    yaml.dump(args_dict, f, sort_keys=False)

# Mirror same content to 2wiki filename (so MDR_embedding_main.py still works)
with open(cfg_2wiki, "w") as f:
    yaml.dump(args_dict, f, sort_keys=False)

print("Wrote config (hotpot):", cfg_hotpot)
print("Mirrored config (2wiki name):", cfg_2wiki)
print("Preview:", yaml.safe_load(open(cfg_hotpot)))

Backed up: /content/drive/MyDrive/final_project/baseline/configs/mdr_embedding/mdr_hotpot_embedding.yml.bak
Backed up: /content/drive/MyDrive/final_project/baseline/configs/mdr_embedding/mdr_2wiki_embedding.yml.bak
Wrote config (hotpot): /content/drive/MyDrive/final_project/baseline/configs/mdr_embedding/mdr_hotpot_embedding.yml
Mirrored config (2wiki name): /content/drive/MyDrive/final_project/baseline/configs/mdr_embedding/mdr_2wiki_embedding.yml
Preview: {'root_dir': '.', 'dataset': 'DATA/HotpotQA/hotpotqa_dev_2017wiki_1000_converted.json', 'model': {'run_id': 'hotpotqa_dev2017wiki_1000_old', 'base_model': 'bert-base-uncased', 'from_checkpoint': 'mdr_best_model_old.pt', 'batch_size': 1, 'max_token_len': 200, 'device': 'cuda', 'save_every': 250000}, 'emb_checkpoint': ''}


In [8]:
# Cell 8 : Patch/wrap OLD checkpoint + update BOTH configs to point to patched checkpoint
import os
import numpy as np
import torch
import yaml
from collections import OrderedDict

cfg_hotpot = os.path.join(BASE_DIR, "configs", "mdr_embedding", "mdr_hotpot_embedding.yml")
cfg_2wiki  = os.path.join(BASE_DIR, "configs", "mdr_embedding", "mdr_2wiki_embedding.yml")

old_ckpt_path = os.path.join(BASE_DIR, "mdr_best_model_old.pt")
assert os.path.isfile(old_ckpt_path), f"Missing checkpoint: {old_ckpt_path}"

def safe_torch_load(path, map_location="cpu"):
    try:
        import torch.serialization
        torch.serialization.add_safe_globals([np.core.multiarray.scalar, np.dtype])
    except Exception:
        pass
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def looks_like_state_dict(obj):
    if not isinstance(obj, (dict, OrderedDict)) or len(obj) == 0:
        return False
    k0 = next(iter(obj.keys()))
    v0 = obj[k0]
    return isinstance(k0, str) and torch.is_tensor(v0)

def extract_state_dict_any(ckpt_obj):
    if isinstance(ckpt_obj, dict):
        if "model_state_dict" in ckpt_obj and isinstance(ckpt_obj["model_state_dict"], (dict, OrderedDict)):
            return ckpt_obj["model_state_dict"]
        if "state_dict" in ckpt_obj and isinstance(ckpt_obj["state_dict"], (dict, OrderedDict)):
            return ckpt_obj["state_dict"]
        if "model" in ckpt_obj and isinstance(ckpt_obj["model"], (dict, OrderedDict)):
            return ckpt_obj["model"]
        if looks_like_state_dict(ckpt_obj):
            return ckpt_obj
    if looks_like_state_dict(ckpt_obj):
        return ckpt_obj
    raise TypeError(f"Unrecognized checkpoint format: {type(ckpt_obj)}")

def normalize_state_dict_keys(sd: dict) -> dict:
    out = {}
    for k, v in sd.items():
        if k.startswith("module."):
            k = k[len("module."):]
        if k.startswith("."):
            k = k[1:]

        if k.startswith("encoder.") or k.startswith("project."):
            out[k] = v
            continue

        if k.startswith("embeddings.") or k.startswith("encoder.") or k.startswith("pooler."):
            out["encoder." + k] = v
            continue

        out[k] = v
    return out

# Load old checkpoint
ckpt_obj = safe_torch_load(old_ckpt_path, map_location="cpu")
sd_raw = extract_state_dict_any(ckpt_obj)
sd_norm = normalize_state_dict_keys(sd_raw)

# Build model (old backbone)
BASE_MODEL = "bert-base-uncased"
tok, cfg = load_tokenizer(model_name=BASE_MODEL)
model = Retriever_inf(cfg, base_model=BASE_MODEL)
model_sd = model.state_dict()

# Make strict-loadable
patched_sd = {}
missing = []
for k in model_sd.keys():
    if k in sd_norm:
        patched_sd[k] = sd_norm[k]
    else:
        patched_sd[k] = model_sd[k]
        missing.append(k)

unexpected = [k for k in sd_norm.keys() if k not in model_sd]

print("Missing filled from init:", len(missing))
print("Unexpected dropped:", len(unexpected))

# Verify strict
model.load_state_dict(patched_sd, strict=True)
print("Strict load verification: OK")

# Save repo-compatible checkpoint
patched_path = os.path.join(BASE_DIR, "mdr_best_model_old_patched.pt")
torch.save({"model_state_dict": patched_sd}, patched_path)
print("Saved patched checkpoint:", patched_path)

# Update BOTH configs
for cfg_file in [cfg_hotpot, cfg_2wiki]:
    cfg = yaml.safe_load(open(cfg_file, "r"))
    cfg["model"]["from_checkpoint"] = os.path.basename(patched_path)
    with open(cfg_file, "w") as f:
        yaml.dump(cfg, f, sort_keys=False)
    print("Updated:", cfg_file, "-> from_checkpoint =", cfg["model"]["from_checkpoint"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Missing filled from init: 0
Unexpected dropped: 1
Strict load verification: OK
Saved patched checkpoint: /content/drive/MyDrive/final_project/baseline/mdr_best_model_old_patched.pt
Updated: /content/drive/MyDrive/final_project/baseline/configs/mdr_embedding/mdr_hotpot_embedding.yml -> from_checkpoint = mdr_best_model_old_patched.pt
Updated: /content/drive/MyDrive/final_project/baseline/configs/mdr_embedding/mdr_2wiki_embedding.yml -> from_checkpoint = mdr_best_model_old_patched.pt


In [9]:
# Cell 9 (REPLACE): Run embedding using HOTPOT config (but 2wiki file is mirrored too)
import os, json
import numpy as np
import torch

from KGP.MDR.tokenizer import load_tokenizer
from KGP.KG.mdr_encoder import Retriever_inf
from KGP.KG.train import run
from KGP.LLMs.Mistral.quantize_mistral_mlx import load_config

def safe_torch_load(path, map_location="cpu"):
    try:
        import torch.serialization
        torch.serialization.add_safe_globals([np.core.multiarray.scalar, np.dtype])
    except Exception:
        pass
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

args = load_config("./configs/mdr_embedding/mdr_hotpot_embedding.yml")

data_path = os.path.join(args["root_dir"], args["dataset"])
raw_documents_data = json.load(open(data_path, "r"))
print("Loaded dataset:", data_path, "records =", len(raw_documents_data))

tokenizer, config = load_tokenizer(model_name=args["model"]["base_model"])
model = Retriever_inf(config, base_model=args["model"]["base_model"])

ckpt_path = os.path.join(args["root_dir"], args["model"]["from_checkpoint"])
model_checkpoint = safe_torch_load(ckpt_path, map_location="cpu")
print("Loading model from checkpoint:", ckpt_path)
model.load_state_dict(model_checkpoint["model_state_dict"])  # repo-style
print("Model loaded successfully.")

run(raw_documents_data, model, tokenizer, args)

Loaded dataset: ./DATA/HotpotQA/hotpotqa_dev_2017wiki_1000_converted.json records = 1000
Loading model from checkpoint: ./mdr_best_model_old_patched.pt
Model loaded successfully.
Parsing raw data...


100%|██████████| 1000/1000 [00:00<00:00, 15831.27it/s]


Finished...
Raw data saved...
No checkpoint found. Starting from scratch...


 57%|█████▋    | 250015/441795 [44:47<2:09:13, 24.74it/s]

Checkpoint saved at 250000...


100%|██████████| 441795/441795 [1:19:31<00:00, 92.58it/s]


441795 batches processed. (Total: 441795)


In [10]:
# Cell 10 (REPLACE): Verify saved outputs (hotpot config)
import os
import numpy as np
import yaml

cfg_path = os.path.join(BASE_DIR, "configs", "mdr_embedding", "mdr_hotpot_embedding.yml")
run_id = yaml.safe_load(open(cfg_path, "r"))["model"]["run_id"]

out_dir = os.path.join(BASE_DIR, "DATA", "KG", "emb", f"emb_{run_id}")

passages_path = os.path.join(out_dir, "passages.json")
emb_path = os.path.join(out_dir, "passage.npy")
cfg_saved = os.path.join(out_dir, "config.yml")

print("Output folder:", out_dir)
print("passages.json exists:", os.path.isfile(passages_path))
print("passage.npy exists:", os.path.isfile(emb_path))
print("config.yml exists:", os.path.isfile(cfg_saved))

embs = np.load(emb_path)
print("Embeddings shape:", embs.shape)
print("First row (first 5 vals):", embs[0][:5])

Output folder: /content/drive/MyDrive/final_project/baseline/DATA/KG/emb/emb_hotpotqa_dev2017wiki_1000_old
passages.json exists: True
passage.npy exists: True
config.yml exists: True
Embeddings shape: (441795, 768)
First row (first 5 vals): [ 0.28142673  1.439111   -0.93503183 -1.6543425   0.30692104]


In [11]:
# Cell 11: Locate and load HotpotQA val_with_neg_v0.json (repo-style)
import os, json

val_path = os.path.join(BASE_DIR, "DATA", "HotpotQA", "MDR", "val_with_neg_v0.json")
assert os.path.isfile(val_path), f"Missing val file: {val_path}"

def load_val_jsonl_filtered(path):
    # Keep only samples that have at least 2 negatives (required by evaluation)
    data = []
    with open(path, "r") as f:
        for line in f:
            d = json.loads(line)
            if "neg_paras" in d and len(d["neg_paras"]) >= 2:
                data.append(d)
    return data

val_data = load_val_jsonl_filtered(val_path)
print("Loaded val:", val_path)
print("Num eval samples (>=2 neg):", len(val_data))
print("Example keys:", list(val_data[0].keys()))

Loaded val: /content/drive/MyDrive/final_project/baseline/DATA/HotpotQA/MDR/val_with_neg_v0.json
Num eval samples (>=2 neg): 7405
Example keys: ['question', 'answers', 'type', 'pos_paras', 'neg_paras', '_id']


In [12]:
# Cell 12: Load the EXACT embedding model (Retriever_inf) + patched OLD checkpoint
import os, yaml
import torch
import numpy as np

cfg_hotpot = os.path.join(BASE_DIR, "configs", "mdr_embedding", "mdr_hotpot_embedding.yml")
assert os.path.isfile(cfg_hotpot), f"Missing config: {cfg_hotpot}"

emb_args = yaml.safe_load(open(cfg_hotpot, "r"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_model = emb_args["model"]["base_model"]
ckpt_name = emb_args["model"]["from_checkpoint"]
ckpt_path = os.path.join(BASE_DIR, ckpt_name)

assert os.path.isfile(ckpt_path), f"Missing checkpoint used for embedding: {ckpt_path}"

print("Embedding config:", cfg_hotpot)
print("base_model:", base_model)
print("checkpoint:", ckpt_path)
print("device:", device)

# Same tokenizer/config function from repo
tokenizer, config = load_tokenizer(model_name=base_model)

# Same model class used for embeddings
model_inf = Retriever_inf(config, base_model=base_model).to(device)
model_inf.eval()

def safe_torch_load(path, map_location="cpu"):
    try:
        import torch.serialization
        torch.serialization.add_safe_globals([np.core.multiarray.scalar, np.dtype])
    except Exception:
        pass
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

ckpt_obj = safe_torch_load(ckpt_path, map_location="cpu")
assert isinstance(ckpt_obj, dict) and "model_state_dict" in ckpt_obj, "Checkpoint must contain model_state_dict"

# Strict load (should work, because we patched it)
missing, unexpected = model_inf.load_state_dict(ckpt_obj["model_state_dict"], strict=True)
print("Strict load OK. missing:", len(missing), "unexpected:", len(unexpected))

Embedding config: /content/drive/MyDrive/final_project/baseline/configs/mdr_embedding/mdr_hotpot_embedding.yml
base_model: bert-base-uncased
checkpoint: /content/drive/MyDrive/final_project/baseline/mdr_best_model_old_patched.pt
device: cuda
Strict load OK. missing: 0 unexpected: 0


In [13]:
# Cell 13: Dataset + collate (same logic as your Evaluate_MDR notebook)
import random
import numpy as np
from torch.utils.data import Dataset

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

class HotpotQANeg(Dataset):
    def __init__(self, data, tokenizer, args, train: bool):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_len = args["max_len"]
        self.max_q_len = args["max_q_len"]
        self.max_q_sp_len = args["max_q_sp_len"]
        self.data = data
        self.train = train

    def encode_chunk_pair(self, t1, t2, max_len):
        return self.tokenizer(
            text=t1, text_pair=t2, max_length=max_len,
            return_tensors="pt", padding=True, truncation=True
        )

    def encode_chunk(self, t, max_len):
        return self.tokenizer(
            text=t, max_length=max_len,
            return_tensors="pt", padding=True, truncation=True
        )

    def __getitem__(self, index):
        d = self.data[index]
        question = d["question"]
        if question.endswith("?"):
            question = question[:-1]

        if d["type"] == "comparison":
            random.shuffle(d["pos_paras"])
            start_para, bridge_para = d["pos_paras"][0], d["pos_paras"][1]
        else:
            for para in d["pos_paras"]:
                if para["title"] != d["bridge"]:
                    start_para = para
                else:
                    bridge_para = para

        if self.train:
            random.shuffle(d["neg_paras"])

        c1_enc = self.encode_chunk_pair(start_para["title"].strip(), start_para["text"].strip(), self.max_len)
        c2_enc = self.encode_chunk_pair(bridge_para["title"].strip(), bridge_para["text"].strip(), self.max_len)

        n1_enc = self.encode_chunk_pair(d["neg_paras"][0]["title"].strip(), d["neg_paras"][0]["text"].strip(), self.max_len)
        n2_enc = self.encode_chunk_pair(d["neg_paras"][1]["title"].strip(), d["neg_paras"][1]["text"].strip(), self.max_len)

        q_enc = self.encode_chunk(question, max_len=self.max_q_len)
        q_c1_enc = self.encode_chunk_pair(question, start_para["text"].strip(), self.max_q_sp_len)

        return {
            "q_enc": q_enc, "q_c1_enc": q_c1_enc,
            "c1_enc": c1_enc, "c2_enc": c2_enc,
            "n1_enc": n1_enc, "n2_enc": n2_enc
        }

    def __len__(self):
        return len(self.data)

def collate_tokens(values, pad_idx, eos_idx=None, left_pad=False, move_eos_to_beginning=False):
    if len(values[0].size()) > 1:
        values = [v.view(-1) for v in values]
    size = max(v.size(0) for v in values)
    res = values[0].new(len(values), size).fill_(pad_idx)

    def copy_tensor(src, dst):
        assert dst.numel() == src.numel()
        if move_eos_to_beginning:
            assert src[-1] == eos_idx
            dst[0] = eos_idx
            dst[1:] = src[:-1]
        else:
            dst.copy_(src)

    for i, v in enumerate(values):
        copy_tensor(v, res[i][size - len(v):] if left_pad else res[i][:len(v)])
    return res

def Dataset_collate(samples):
    if len(samples) == 0:
        return {}
    batch = {
        "q_enc_btz": collate_tokens([s["q_enc"]["input_ids"].view(-1) for s in samples], 0),
        "q_mask": collate_tokens([s["q_enc"]["attention_mask"][0] for s in samples], 0),

        "q_c1_enc_btz": collate_tokens([s["q_c1_enc"]["input_ids"].view(-1) for s in samples], 0),
        "q_c1_mask": collate_tokens([s["q_c1_enc"]["attention_mask"][0] for s in samples], 0),

        "c1_enc_btz": collate_tokens([s["c1_enc"]["input_ids"].view(-1) for s in samples], 0),
        "c1_mask": collate_tokens([s["c1_enc"]["attention_mask"][0] for s in samples], 0),

        "c2_enc_btz": collate_tokens([s["c2_enc"]["input_ids"].view(-1) for s in samples], 0),
        "c2_mask": collate_tokens([s["c2_enc"]["attention_mask"][0] for s in samples], 0),

        "n1_enc_btz": collate_tokens([s["n1_enc"]["input_ids"].view(-1) for s in samples], 0),
        "n1_mask": collate_tokens([s["n1_enc"]["attention_mask"][0] for s in samples], 0),

        "n2_enc_btz": collate_tokens([s["n2_enc"]["input_ids"].view(-1) for s in samples], 0),
        "n2_mask": collate_tokens([s["n2_enc"]["attention_mask"][0] for s in samples], 0),
    }
    return batch

In [14]:
# Cell 14: Exact MRR evaluation using Retriever_inf (same scoring logic as Evaluate_MDR)
import torch
import numpy as np
from tqdm import tqdm

@torch.no_grad()
def encode_with_inf(model_inf, input_ids, attention_mask):
    # model_inf forward returns embeddings
    return model_inf(input_ids, attention_mask)

@torch.no_grad()
def forward_batch_inf(model_inf, batch):
    # Produce the same embedding dict structure used by mhop_eval
    q_emb    = encode_with_inf(model_inf, batch["q_enc_btz"],    batch["q_mask"])
    q_c1_emb = encode_with_inf(model_inf, batch["q_c1_enc_btz"], batch["q_c1_mask"])

    c1_emb = encode_with_inf(model_inf, batch["c1_enc_btz"], batch["c1_mask"])
    c2_emb = encode_with_inf(model_inf, batch["c2_enc_btz"], batch["c2_mask"])

    n1_emb = encode_with_inf(model_inf, batch["n1_enc_btz"], batch["n1_mask"])
    n2_emb = encode_with_inf(model_inf, batch["n2_enc_btz"], batch["n2_mask"])

    return {
        "q_emb": q_emb, "q_c1_emb": q_c1_emb,
        "c1_emb": c1_emb, "c2_emb": c2_emb,
        "n1_emb": n1_emb, "n2_emb": n2_emb
    }

@torch.no_grad()
def mhop_eval_exact(embs):
    # Same logic you used earlier (repo-faithful)
    c_embs = torch.cat([embs["c1_emb"], embs["c2_emb"]], dim=0)  # (2B) x D
    n_embs = torch.cat([embs["n1_emb"].unsqueeze(1), embs["n2_emb"].unsqueeze(1)], dim=1)  # B x 2 x D

    scores_1 = torch.mm(embs["q_emb"], c_embs.t())  # B x 2B
    n_scores_1 = torch.bmm(embs["q_emb"].unsqueeze(1), n_embs.permute(0, 2, 1)).squeeze(1)  # B x 2

    scores_2 = torch.mm(embs["q_c1_emb"], c_embs.t())  # B x 2B
    # repo uses q_emb here (not q_c1_emb)
    n_scores_2 = torch.bmm(embs["q_emb"].unsqueeze(1), n_embs.permute(0, 2, 1)).squeeze(1)  # B x 2

    bsize = embs["q_emb"].size(0)
    scores_1_mask = torch.cat([torch.zeros(bsize, bsize), torch.eye(bsize)], dim=1).to(embs["q_emb"].device)
    scores_1 = scores_1.float().masked_fill(scores_1_mask.bool(), float("-inf")).type_as(scores_1)

    scores_1 = torch.cat([scores_1, n_scores_1], dim=1)  # B x (2B+2)
    scores_2 = torch.cat([scores_2, n_scores_2], dim=1)  # B x (2B+2)

    target_1 = torch.arange(bsize).to(embs["q_emb"].device)
    target_2 = torch.arange(bsize).to(embs["q_emb"].device) + bsize

    ranked_1_hop = scores_1.argsort(dim=1, descending=True)
    ranked_2_hop = scores_2.argsort(dim=1, descending=True)
    idx2ranked_1 = ranked_1_hop.argsort(dim=1)
    idx2ranked_2 = ranked_2_hop.argsort(dim=1)

    rrs_1, rrs_2 = [], []
    for t, idx2ranked in zip(target_1, idx2ranked_1):
        rrs_1.append(1.0 / (idx2ranked[t].item() + 1))
    for t, idx2ranked in zip(target_2, idx2ranked_2):
        rrs_2.append(1.0 / (idx2ranked[t].item() + 1))

    return rrs_1, rrs_2

@torch.no_grad()
def eval_mrr_inf(model_inf, dataloader, device):
    model_inf.eval()
    rrs_1_all, rrs_2_all = [], []
    for batch in tqdm(dataloader, desc="Eval (OLD embedding model)"):
        # Move to GPU
        for k in batch:
            batch[k] = batch[k].to(device)

        embs = forward_batch_inf(model_inf, batch)
        r1, r2 = mhop_eval_exact(embs)
        rrs_1_all += r1
        rrs_2_all += r2

    return float(np.mean(rrs_1_all)), float(np.mean(rrs_2_all))

In [15]:
# Cell 15: Evaluate OLD embedding model on HotpotQA val_with_neg_v0.json
from torch.utils.data import DataLoader
import torch

# IMPORTANT: Use the SAME token length settings as training MDR.yml for a fair comparison.
# We load MDR.yml for these max_len/max_q_len/max_q_sp_len values.
import yaml, os
mdr_cfg = yaml.safe_load(open(os.path.join(BASE_DIR, "configs", "MDR.yml"), "r"))

eval_args = {
    "max_len": mdr_cfg["max_len"],
    "max_q_len": mdr_cfg["max_q_len"],
    "max_q_sp_len": mdr_cfg["max_q_sp_len"],
}

print("Eval token lengths:", eval_args)

ds = HotpotQANeg(val_data, tokenizer, eval_args, train=False)

# Try batch sizes to avoid OOM
for bsz in [32, 16, 8, 4, 2, 1]:
    try:
        dl = DataLoader(ds, batch_size=bsz, shuffle=False, num_workers=0, pin_memory=True, collate_fn=Dataset_collate)
        mrr1, mrr2 = eval_mrr_inf(model_inf, dl, device=device)
        print(f"BSZ={bsz} -> MRR_1={mrr1:.6f} | MRR_2={mrr2:.6f} | AVG={(mrr1+mrr2)/2:.6f}")
        break
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("OOM at bsz =", bsz, "-> trying smaller...")
            torch.cuda.empty_cache()
            continue
        raise

Eval token lengths: {'max_len': 200, 'max_q_len': 64, 'max_q_sp_len': 256}


Eval (OLD embedding model): 100%|██████████| 232/232 [04:06<00:00,  1.06s/it]

BSZ=32 -> MRR_1=0.951877 | MRR_2=0.925885 | AVG=0.938881
